<a href="https://colab.research.google.com/github/robin8a/sse_etl/blob/main/GLV_ETL_Providers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Environment

## ETL to load items from providers

Help to create a script  to extract, transfrom and load (ETL) data from a google drive google sheet file, another google sheet as a template, and output file also a google sheet file.  

## Help to update the script with:

- Also in drive params.json with the following structure

```json
{
  "xlsx_file_provider": "",
  "xlsx_file_provider_sheets": [""],
  "xlsx_file_provider_row_to_start": "",   
  "xlsx_file_template": "",
  "xlsx_file_output": "",

}
```

- The parameter "xlsx_file_provider_row_to_start" is the row that contains the columuns to match with
- The parameter "xlsx_file_provider_has_subcategory" if is true, means there are row that has subcategory value in the row not the N columns as is in "xlsx_file_provider_row_to_start"
- Allow me to match "xlsx_file_provider" columns with the "xlsx_file_template" columnms, also allow me to check if the column does exist an added to end with "NOT_MATH_<column_name>
- The source file also includes and embed "image" column that needs to be extracted and upload to AWS S3 bucket.





In [3]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.8 MB/s eta 0:00:00


In [4]:
import json
import pandas as pd
import gspread
from google.colab import auth
import boto3
from io import StringIO
import re
import os
import requests # For downloading images
import mimetypes # To guess image file types
from google.auth import default # Import default

# Install required libraries
!pip install --upgrade gspread pandas boto3 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.


## Google Drive Authentication

To write to Google Sheets, you need to authenticate this Colab notebook with your Google account. Run the following cell and follow the instructions to grant access.

## AWS S3 Configuration

To upload images to AWS S3, you will need to provide your AWS credentials. It's recommended to store these as Colab secrets for security. Click on the '🔑' icon in the left sidebar, then add your `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`.

You will also need to specify your AWS region and S3 bucket name.

In [29]:
# Import the Colab userdata library
from google.colab import userdata

# Retrieve AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = userdata.get('sse_aws_access_key_id')
AWS_SECRET_ACCESS_KEY = userdata.get('sse_aws_secret_access_key')
AWS_REGION = 'your-aws-region' # e.g., 'us-east-1', update this
AWS_S3_BUCKET_NAME = 'your-s3-bucket-name' # update this with your S3 bucket name

# Initialize S3 client
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

## Configuration Parameters (params.json)

Below is the structure for your `params.json` file. Please update the placeholder values with your actual Google Sheet file IDs or names, and other relevant parameters. You can run this cell to create a `params.json` file in your Colab environment or manually upload one to `/content/params.json`.

In [30]:
params_content = {
  "xlsx_file_provider": "/content/ListadoGVS(13.04.26).xlsx", # Path to the provider Excel file
  "xlsx_file_provider_sheets": ["OUTLET"], # Array of sheet names to process from the provider Excel, e.g., ["Sheet1", "Sheet2"]
  "txt_file_match": "/content/match.txt", # Path to the text file for column matching
  "csv_file_output_patterns": "/content/output_pattern.csv", # Path for the output CSV file (renamed to reflect patterns)
  "csv_file_output_match": "/content/output_match.csv", # Path for the output CSV file match.txt
  "s3_bucket": "ssebeapief4a7d425930451b867d315da2ab24c9cba30-dev", # S3 bucket name for images
  "s3_bucket_prefix": "providers_image/", # S3 prefix (folder) for images
  "s3_image_start_with": "glv_" # Prefix for uploaded image filenames
}

# Save this to a params.json file
with open('params.json', 'w') as f:
    json.dump(params_content, f, indent=2)

print("params.json created. Please update its content in the file browser or directly in this cell.")

params.json created. Please update its content in the file browser or directly in this cell.


## Load Parameters

Run this cell to load the parameters from `params.json`.

In [31]:
with open('params.json', 'r') as f:
    params = json.load(f)

# Display loaded parameters for verification
print(json.dumps(params, indent=2))

{
  "xlsx_file_provider": "/content/ListadoGVS(13.04.26).xlsx",
  "xlsx_file_provider_sheets": [
    "OUTLET"
  ],
  "txt_file_match": "/content/match.txt",
  "csv_file_output_patterns": "/content/output_pattern.csv",
  "csv_file_output_match": "/content/output_match.csv",
  "s3_bucket": "ssebeapief4a7d425930451b867d315da2ab24c9cba30-dev",
  "s3_bucket_prefix": "providers_image/",
  "s3_image_start_with": "glv_"
}


# Patterns Detected
Categorization by Sheet: Each sheet (represented by a CSV file) acts as a high-level Category (e.g., CCTV, Almacenamiento, Incendio).

Brand Identification: Brands are typically listed in a row where the first column contains the brand name (e.g., HIKVISION, ZKTECO), but the product code column is empty.

Subcategory/Header Rows: Rows starting with the text "Codigo" in the second column serve as subcategory headers. The text immediately to the right of "Codigo" defines the Subcategory (e.g., -IP NVR SERIE 7600, DISCOS DUROS).

Product Data Structure: Valid product rows always contain a unique identifier in the Code column and corresponding pricing/description data.

Column 1: Product Code

Column 2: Description (often includes "Existencia Única" notes)

Column 3 & 4: Base Price and Price with IVA

Column 5 - 8: Offer and group metadata

In [32]:
import pandas as pd
import re

def process_gvs_list(params):
    input_excel = params['xlsx_file_provider']
    output_csv = params['csv_file_output_patterns']
    sheets_to_process = params['xlsx_file_provider_sheets']

    # Load the Excel file
    xl = pd.ExcelFile(input_excel)
    all_products = []

    # Iterate through each specified sheet (Category)
    for sheet_name in sheets_to_process:
        # Skip the index sheet if it's explicitly mentioned in the sheet names (though usually handled by 'INDICE' check below)
        if 'INDICE' in sheet_name.upper():
            continue

        print(f"Processing category: {sheet_name}...")

        # Read sheet without headers to handle custom layout where metadata rows are interspersed
        df = xl.parse(sheet_name, header=None)

        current_brand = "Unknown"
        current_subcategory = "General"

        for index, row in df.iterrows():
            # Clean row data and convert to strings
            clean_row = [str(val).strip() if pd.notnull(val) else "" for val in row]

            # 1. Detect Brand Rows
            # (Column 0 has text, but the 'Codigo' column 1 is empty)
            if clean_row[0] != "" and clean_row[1] == "" and clean_row[2] == "":
                current_brand = clean_row[0]
                continue

            # 2. Detect Subcategory/Header Rows
            # (Column 1 contains the word "Codigo")
            if clean_row[1].lower() == "codigo":
                # The subcategory name is usually in the next column
                current_subcategory = clean_row[2] if clean_row[2] != "" else current_subcategory
                continue

            # 3. Detect Product Rows
            # (Column 1 is not empty, and there is a price in column 3 or 4)
            # This logic assumes the fixed column indices correspond to product data after brand/subcategory parsing
            if clean_row[1] != "" and (clean_row[3] != "" or clean_row[4] != ""):
                product_data = {
                    'Category': sheet_name,
                    'Brand': current_brand,
                    'Subcategory': current_subcategory,
                    'Code': clean_row[1],
                    'Description': clean_row[2],
                    'Price': clean_row[3],
                    'Price_with_IVA': clean_row[4],
                    'Discount': clean_row[5],
                    'Offer_Price_with_IVA': clean_row[6],
                    'Offer_Validity': clean_row[7],
                    'Group': clean_row[8]
                }
                all_products.append(product_data)

    # Create a DataFrame and export to CSV
    df_final = pd.DataFrame(all_products)
    df_final.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\nSuccess! Exported {len(df_final)} products to {output_csv}")

# --- Execution ---
# Call the function with params
process_gvs_list(params)

Processing category: OUTLET...

Success! Exported 39 products to /content/output_pattern.csv


# Matching

In [35]:
def apply_column_matching(params):
    input_patterns_csv = params['csv_file_output_patterns']
    match_file = params['txt_file_match']
    output_match_csv = params['csv_file_output_match']

    # 1. Load the patterns DataFrame
    try:
        df_patterns = pd.read_csv(input_patterns_csv)
        print(f"Successfully loaded {len(df_patterns)} rows from {input_patterns_csv}")
    except FileNotFoundError:
        print(f"Error: {input_patterns_csv} not found. Please ensure the previous step ran successfully.")
        return

    # 2. Load ordered column definitions from match.txt
    # Each mapping is 'TargetOutputColumnName=SourceProviderColumnName'
    # If SourceProviderColumnName is 'NOT_COLUMN', it creates an empty column.
    ordered_output_columns = [] # List of (target_output_name, source_name)
    try:
        with open(match_file, 'r') as f:
            content = f.read().strip() # Read entire content
            if not content:
                print(f"Warning: {match_file} is empty. No custom column matching will be applied.")
                # If match.txt is empty, default to original columns without renaming
                df_patterns.to_csv(output_match_csv, index=False, encoding='utf-8-sig')
                print(f"Exported data without matching to {output_match_csv}")
                return

            # Split the single line content by pipe to get individual mappings
            raw_mappings = content.split('|')

            for raw_map in raw_mappings:
                raw_map = raw_map.strip()
                if '=' in raw_map:
                    # target_output_name is the column name for the final output (csv_file_output_match)
                    # source_name is the column name from the input patterns file (csv_file_output_patterns)
                    target_output_name, source_name = raw_map.split('=', 1)
                    ordered_output_columns.append((target_output_name.strip(), source_name.strip()))
                else:
                    print(f"Warning: Invalid mapping format '{raw_map}' in {match_file}. Skipping. Expected format: 'TargetColumn=SourceColumn'")
        print(f"Loaded {len(ordered_output_columns)} column definitions from {match_file}")

    except FileNotFoundError:
        print(f"Warning: {match_file} not found. No custom column matching will be applied.")
        df_patterns.to_csv(output_match_csv, index=False, encoding='utf-8-sig')
        print(f"Exported data without matching to {output_match_csv}")
        return

    # 3. Apply mappings and handle unmatched columns
    df_final_matched = pd.DataFrame()
    used_provider_cols = set() # To track which provider columns have been used

    for target_output_name, source_name in ordered_output_columns:
        if source_name == 'NOT_COLUMN':
            # Create an empty column with the specified target_output_name
            df_final_matched[target_output_name] = [None] * len(df_patterns) # Use None for empty cells
            print(f"Created empty output column: '{target_output_name}'")
        elif source_name in df_patterns.columns:
            df_final_matched[target_output_name] = df_patterns[source_name]
            used_provider_cols.add(source_name)
        else:
            # Source column not found in provider data, create an empty column for the target
            print(f"Warning: Source column '{source_name}' for output '{target_output_name}' not found in input. Creating empty column.")
            df_final_matched[target_output_name] = [None] * len(df_patterns)

    # 4. Handle provider columns from df_patterns that were not explicitly mapped in match.txt
    # These will be added to the end of the DataFrame with a 'NOT_MATCH_' prefix
    for col in df_patterns.columns:
        if col not in used_provider_cols and col not in [item[1] for item in ordered_output_columns if item[1] != 'NOT_COLUMN']:
            # Check if the column is already added as an empty 'NOT_COLUMN' based on its name
            # Or if its target name in the mappings already exists as a column name.
            if f'NOT_MATCH_{col}' not in df_final_matched.columns:
                df_final_matched[f'NOT_MATCH_{col}'] = df_patterns[col]
                print(f"Added unmatched provider column '{col}' as 'NOT_MATCH_{col}'")

    # 5. Export the final matched DataFrame to csv_file_output_match
    df_final_matched.to_csv(output_match_csv, index=False, encoding='utf-8-sig')
    print(f"\nSuccess! Exported {len(df_final_matched)} products with column matching to {output_match_csv}")

# --- Execution ---
# Call the function to apply column matching and generate the output_match.csv
apply_column_matching(params)

Successfully loaded 39 rows from /content/output_pattern.csv
ITEM_SUB_CATEGORY' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
GENERAL_PROFIT_PERCENTAGE' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
NAME_TO_SHOW' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
OWN_NAME' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
OWN_INVENTORY_CODE' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
PROVIDER_CATEGORY' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
PROVIDER_BRAND' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
PROVIDER_SUBCATEGORY' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
PROVIDER_INVENTORY_CODE' in /content/match.txt. Skipping. Expected format: 'TargetColumn=SourceColumn'
PROVIDER_DESCRIPTION' in /content/match.txt. Skipping. Expected f